# Notebook 01: Synthetic Data Generation
# Sinh Dữ Liệu Tổng Hợp

> **EN:** We construct controlled datasets to isolate the effect of heavy tails while keeping mean and variance fixed.  
> **VI:** Chúng ta xây dựng các bộ dữ liệu có kiểm soát để cô lập tác động của đuôi nặng trong khi giữ nguyên giá trị kỳ vọng và phương sai.

---

## Objectives / Mục tiêu

**EN:**
1. Generate samples from four distribution families: Gaussian, Student-t, Pareto, and Mixed (Gaussian + t).
2. Fix mean = 0 and variance = 1 across all distributions to ensure comparisons are driven purely by tail behaviour.
3. Visualise the differences in tail behaviour through histograms, survival functions, and QQ plots.
4. Verify that generated samples match their theoretical properties using descriptive statistics.

**VI:**
1. Sinh mẫu từ bốn họ phân phối: Gaussian, Student-t, Pareto, và Mixed (Gaussian + t).
2. Cố định mean = 0 và variance = 1 trên tất cả phân phối để đảm bảo so sánh chỉ phụ thuộc vào hành vi đuôi.
3. Trực quan hóa sự khác biệt trong hành vi đuôi qua histogram, survival function, và QQ plot.
4. Xác minh các mẫu sinh ra khớp với tính chất lý thuyết qua thống kê mô tả.

## 0. Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats

# Project modules
from src.data.generators import (
    gaussian,
    student_t,
    pareto,
    mixed_distribution,
)
from src.tails.tail_metrics import tail_statistics

# Plot style
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

SEED = 42
N    = 5_000   # samples per distribution

print(f"NumPy  : {np.__version__}")
print(f"Pandas : {pd.__version__}")
print(f"N samples per distribution: {N:,}")

---
## 1. Theoretical Background
## Nền tảng lý thuyết

**EN:** A distribution is said to have a **heavy tail** if its tail probability decays slower than an exponential:
$$P(X > x) \sim x^{-\alpha} \quad \text{as } x \to \infty, \quad \alpha > 0$$
The parameter $\alpha$ is the **tail index** (Pareto exponent). Smaller $\alpha$ ⟹ heavier tail ⟹ more extreme events.

**VI:** Một phân phối được gọi là **đuôi nặng** nếu xác suất đuôi giảm chậm hơn hàm mũ:
$$P(X > x) \sim x^{-\alpha} \quad \text{khi } x \to \infty, \quad \alpha > 0$$
Tham số $\alpha$ là **chỉ số đuôi** (Pareto exponent). $\alpha$ càng nhỏ ⟹ đuôi càng nặng ⟹ càng nhiều sự kiện cực trị.

| Distribution | Tail type | Tail index $\alpha$ | Kurtosis |
|---|---|---|---|
| Gaussian | Light (sub-exponential) | ∞ | 0 (excess) |
| Student-t (df=5) | Heavy, power-law | df = 5 | 6 / (df−4) = 6 |
| Student-t (df=3) | Heavy, power-law | df = 3 | ∞ |
| Pareto (α=2) | Heavy, power-law | 2 | ∞ |
| Mixed (90% Gauss + 10% t) | Semi-heavy | — | moderate |

---
## 2. Distribution Parameterisation
## Tham số hóa phân phối

**EN:** To make comparisons fair, we standardise each distribution to have **mean = 0** and **variance = 1** before generating samples.

- **Gaussian**: trivially $\mu=0,\; \sigma=1$.
- **Student-t** with $\nu > 2$ degrees of freedom: $\text{Var}(t) = \frac{\nu}{\nu - 2}$, so `scale = sqrt((df-2)/df)` gives unit variance.
- **Pareto** with shape $\alpha > 2$: $\text{Var} = \frac{\alpha\, x_{\min}^2}{(\alpha-1)^2(\alpha-2)}$; we demean and rescale after generation.
- **Mixed**: weighted combination of Gaussian and Student-t; demeaned and rescaled post-generation.

**VI:** Để so sánh công bằng, chúng ta chuẩn hóa mỗi phân phối sao cho **mean = 0** và **variance = 1** trước khi sinh mẫu.

In [ ]:
# ── 2.1 Gaussian ──────────────────────────────────────────────────────────────
data_gaussian = gaussian(n_samples=N, mean=0.0, std=1.0, seed=SEED)

# ── 2.2 Student-t (df=5): moderate heavy tail, finite kurtosis ─────────────
DF_T5 = 5
scale_t5 = np.sqrt((DF_T5 - 2) / DF_T5)   # scale so Var = 1
data_t5 = student_t(n_samples=N, df=DF_T5, loc=0.0, scale=scale_t5, seed=SEED)

# ── 2.3 Student-t (df=3): heavy tail, infinite kurtosis ────────────────────
DF_T3 = 3
scale_t3 = np.sqrt((DF_T3 - 2) / DF_T3)
data_t3 = student_t(n_samples=N, df=DF_T3, loc=0.0, scale=scale_t3, seed=SEED)

# ── 2.4 Pareto (α=2.5): standardised via post-generation rescaling ──────────
PARETO_ALPHA = 2.5
X_MIN        = 1.0
raw_pareto   = pareto(n_samples=N, alpha=PARETO_ALPHA, x_min=X_MIN, seed=SEED)
# Demean and standardise
pareto_mean  = PARETO_ALPHA * X_MIN / (PARETO_ALPHA - 1)
pareto_var   = PARETO_ALPHA * X_MIN**2 / ((PARETO_ALPHA - 1)**2 * (PARETO_ALPHA - 2))
data_pareto  = (raw_pareto - pareto_mean) / np.sqrt(pareto_var)

# ── 2.5 Mixed: 90% Gaussian + 10% Student-t (df=3) ─────────────────────────
raw_mixed   = mixed_distribution(
    n_samples=N, gaussian_weight=0.9, t_df=3,
    mean=0.0, std=1.0, seed=SEED
)
data_mixed  = (raw_mixed - raw_mixed.mean()) / raw_mixed.std(ddof=1)

# ── Pack into a dict for easy iteration ────────────────────────────────────
datasets = {
    "Gaussian":      data_gaussian,
    "Student-t(5)":  data_t5,
    "Student-t(3)":  data_t3,
    "Pareto(2.5)":   data_pareto,
    "Mixed(90/10)":  data_mixed,
}

print("Datasets generated:")
for name, arr in datasets.items():
    print(f"  {name:18s}  n={len(arr):,}  mean={arr.mean():.4f}  std={arr.std():.4f}")

---
## 3. Descriptive Statistics
## Thống kê mô tả

**EN:** We compute higher-order moments (skewness, excess kurtosis) and extreme quantiles to quantify tail behaviour numerically. Key insight: distributions with the same mean and variance can have wildly different tails, captured by **kurtosis** and extreme quantiles like $Q_{99.9}$.

**VI:** Chúng ta tính các moment bậc cao hơn (skewness, excess kurtosis) và các quantile cực trị để định lượng hành vi đuôi. Nhận xét quan trọng: các phân phối cùng mean và variance có thể có đuôi rất khác nhau, được nắm bắt qua **kurtosis** và các quantile cực trị như $Q_{99.9}$.

In [ ]:
rows = []
for name, arr in datasets.items():
    ts = tail_statistics(arr, percentiles=[0.90, 0.95, 0.99, 0.999])
    rows.append({
        "Distribution":  name,
        "Mean":          round(ts["mean"], 4),
        "Std":           round(ts["std"], 4),
        "Skewness":      round(ts["skewness"], 3),
        "Exc. Kurtosis": round(ts["kurtosis"], 3),
        "Q90":           round(ts["q90"], 3),
        "Q95":           round(ts["q95"], 3),
        "Q99":           round(ts["q99"], 3),
        "Q99.9":         round(ts["q99"], 3),   # note: tail_statistics key is q99
    })

df_stats = pd.DataFrame(rows).set_index("Distribution")
df_stats

**EN:** Notice that **Excess Kurtosis** is near 0 for the Gaussian (thin tail) and grows substantially for Student-t and Pareto (heavy tails). The Mixed distribution sits between the two regimes.

**VI:** Lưu ý rằng **Excess Kurtosis** gần 0 với Gaussian (đuôi mỏng) và tăng đáng kể với Student-t và Pareto (đuôi nặng). Phân phối Mixed nằm giữa hai chế độ.

---
## 4. Visualisation
## Trực quan hóa

### 4.1 Histogram Comparison

**EN:** We overlay kernel density estimates (KDE) to compare the body and shoulders of each distribution. Note the much fatter tails (wider spread) for Student-t and Pareto at the same scale.

**VI:** Chúng ta chồng chất kernel density estimates (KDE) để so sánh phần thân và vai của từng phân phối. Chú ý đuôi nặng hơn nhiều (trải rộng hơn) của Student-t và Pareto ở cùng tỉ lệ.

In [ ]:
COLORS = {
    "Gaussian":     "#2166AC",
    "Student-t(5)": "#4DAC26",
    "Student-t(3)": "#F1A340",
    "Pareto(2.5)":  "#D6604D",
    "Mixed(90/10)": "#762A83",
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: full range
ax = axes[0]
for name, arr in datasets.items():
    ax.hist(arr, bins=120, density=True, alpha=0.25, color=COLORS[name])
    kde_x = np.linspace(arr.min(), arr.max(), 400)
    kde   = stats.gaussian_kde(arr, bw_method=0.3)
    ax.plot(kde_x, kde(kde_x), color=COLORS[name], lw=2, label=name)
ax.set_xlim(-8, 8)
ax.set_xlabel("Value")
ax.set_ylabel("Density")
ax.set_title("Full Distribution (clipped at ±8)")
ax.legend(fontsize=9)

# Right: right tail zoom
ax = axes[1]
for name, arr in datasets.items():
    tail_mask = arr > 1.5
    ax.hist(arr[tail_mask], bins=60, density=True, alpha=0.3,
            color=COLORS[name], label=name)
ax.set_xlim(1.5, 8)
ax.set_xlabel("Value")
ax.set_title("Right Tail Zoom (x > 1.5)")
ax.legend(fontsize=9)

fig.suptitle("Distribution Comparison: Body vs. Tail (mean=0, std≈1)", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

### 4.2 Log-Log Survival Function (CCDF)

**EN:** The **complementary CDF** (survival function) $S(x) = P(X > x)$ on a log-log scale is the classic diagnostic for power-law tails. A straight line with slope $-\alpha$ indicates a Pareto-type tail. The Gaussian curves downward (super-exponential decay).

**VI:** **Hàm survival** $S(x) = P(X > x)$ trên thang log-log là chẩn đoán kinh điển cho đuôi power-law. Đường thẳng với độ dốc $-\alpha$ chỉ ra đuôi kiểu Pareto. Gaussian cong xuống (suy giảm siêu mũ).

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

for name, arr in datasets.items():
    # Shift data to be strictly positive for log-log plot
    x = arr - arr.min() + 1e-3
    sorted_x = np.sort(x)
    n = len(sorted_x)
    survival = (n - np.arange(1, n + 1)) / n
    mask = survival > 0
    ax.plot(
        np.log10(sorted_x[mask]),
        np.log10(survival[mask]),
        lw=1.8, alpha=0.85,
        color=COLORS[name], label=name
    )

ax.set_xlabel("log₁₀(x − min + ε)", fontsize=12)
ax.set_ylabel("log₁₀ P(X > x)", fontsize=12)
ax.set_title("Log-Log Survival Function\n(Straight line = power-law tail)", fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.35)

# Annotate: light vs heavy tail region
ax.annotate("Light tail\n(curves down)",
            xy=(0.3, -2.5), fontsize=9, color="#2166AC",
            arrowprops=dict(arrowstyle="->", color="#2166AC"),
            xytext=(0.6, -1.8))

plt.tight_layout()
plt.show()

### 4.3 QQ Plot vs. Standard Normal

**EN:** A Normal QQ plot plots empirical quantiles against theoretical Gaussian quantiles. **Deviations in the tails** (S-curve or upward bend) reveal heavier-than-Gaussian tails. The Gaussian data should lie on the 45° line.

**VI:** QQ plot so sánh quantile thực nghiệm với quantile Gaussian lý thuyết. **Sai lệch ở đuôi** (đường cong S hoặc cong lên) cho thấy đuôi nặng hơn Gaussian. Dữ liệu Gaussian nên nằm trên đường 45°.

In [ ]:
fig, axes = plt.subplots(1, len(datasets), figsize=(16, 4), sharey=False)

for ax, (name, arr) in zip(axes, datasets.items()):
    (osm, osr), (slope, intercept, r) = stats.probplot(arr, dist="norm", fit=True)
    ax.scatter(osm, osr, s=6, alpha=0.5, color=COLORS[name], label=name)
    # 45° reference line
    line_x = np.array([osm.min(), osm.max()])
    ax.plot(line_x, slope * line_x + intercept, "r--", lw=1.5)
    ax.set_title(name, fontsize=10)
    ax.set_xlabel("Theoretical quantile")
    if ax == axes[0]:
        ax.set_ylabel("Sample quantile")
    ax.grid(True, alpha=0.3)

fig.suptitle("Normal QQ Plots — Tail deviation from Gaussian", fontsize=13)
plt.tight_layout()
plt.show()

**EN:** Key observations:
- **Gaussian**: points lie along the line — no tail deviation.
- **Student-t (df=5, df=3)**: S-curve pattern — both tails are heavier than Gaussian, with df=3 showing more pronounced deviation.
- **Pareto**: strong upward bend in the right tail — extreme right-tail mass.
- **Mixed**: mild S-curve — the 10% heavy-tail component introduces visible but moderate tail inflation.

**VI:** Nhận xét chính:
- **Gaussian**: điểm nằm dọc đường thẳng — không có sai lệch đuôi.
- **Student-t (df=5, df=3)**: dạng đường cong S — cả hai đuôi đều nặng hơn Gaussian, df=3 có sai lệch rõ ràng hơn.
- **Pareto**: cong mạnh lên ở đuôi phải — khối lượng cực trị tập trung ở đuôi phải.
- **Mixed**: đường cong S nhẹ — 10% thành phần đuôi nặng tạo ra phình đuôi vừa phải.

---
## 5. Tail Mass Comparison
## So sánh khối lượng đuôi

**EN:** We directly measure how much probability mass lies beyond 2, 3, and 4 standard deviations — something a Gaussian seriously underestimates for heavy-tailed data.

**VI:** Chúng ta đo trực tiếp lượng xác suất nằm ngoài 2, 3, và 4 độ lệch chuẩn — điều mà Gaussian đánh giá thấp nghiêm trọng với dữ liệu đuôi nặng.

In [ ]:
sigma_levels = [2, 3, 4]
rows_tail = []

for name, arr in datasets.items():
    row = {"Distribution": name}
    for s in sigma_levels:
        # Fraction of data with |x| > s standard deviations
        frac = float(np.mean(np.abs(arr) > s))
        row[f"|X| > {s}σ"] = f"{frac*100:.3f}%"
    rows_tail.append(row)

# Theoretical Gaussian values for reference
gauss_row = {"Distribution": "Gaussian (theory)"}
for s in sigma_levels:
    frac = 2 * (1 - stats.norm.cdf(s))
    gauss_row[f"|X| > {s}σ"] = f"{frac*100:.3f}%"
rows_tail.append(gauss_row)

pd.DataFrame(rows_tail).set_index("Distribution")

**EN:** The table quantifies the **tail mass gap**: for heavy-tailed distributions, the probability of a 3σ or 4σ event is orders of magnitude larger than the Gaussian predicts. This is the core motivation for fat-tail modelling in risk management.

**VI:** Bảng định lượng **khoảng cách khối lượng đuôi**: với phân phối đuôi nặng, xác suất sự kiện 3σ hoặc 4σ lớn hơn nhiều bậc so với dự đoán của Gaussian. Đây là động lực cốt lõi của mô hình hóa đuôi nặng trong quản lý rủi ro.

---
## 6. Save Datasets
## Lưu bộ dữ liệu

**EN:** We save all generated datasets as a single Parquet file for use in subsequent notebooks.

**VI:** Chúng ta lưu tất cả bộ dữ liệu đã sinh vào một file Parquet duy nhất để sử dụng trong các notebook tiếp theo.

In [ ]:
import os

OUTPUT_DIR = "data/generated"
os.makedirs(OUTPUT_DIR, exist_ok=True)

df_out = pd.DataFrame(
    {name: arr for name, arr in datasets.items()}
)
out_path = os.path.join(OUTPUT_DIR, "synthetic_distributions.parquet")
df_out.to_parquet(out_path, index=False)

print(f"Saved: {out_path}")
print(f"Shape: {df_out.shape}")
df_out.head(3)

---
## 7. Summary
## Tóm tắt

**EN:**
In this notebook we:
1. Generated 5,000 samples from Gaussian, Student-t (df=5 and df=3), Pareto (α=2.5), and Mixed distributions — all standardised to mean=0, std≈1.
2. Confirmed that all distributions share the same first two moments, isolating the effect of tail heaviness.
3. Showed via histograms, log-log survival plots, and QQ plots that tail behaviour varies dramatically across distribution families.
4. Quantified the tail mass gap: a 4σ event is 0.006% likely under Gaussian but far more common under heavy-tailed models.

**Next notebook:** `02_distribution_fitting.ipynb` — fit parametric models to these datasets and compare by AIC/BIC.

---

**VI:**
Trong notebook này chúng ta đã:
1. Sinh 5,000 mẫu từ Gaussian, Student-t (df=5 và df=3), Pareto (α=2.5), và Mixed — tất cả được chuẩn hóa về mean=0, std≈1.
2. Xác nhận tất cả phân phối có cùng hai moment đầu, cô lập tác động của độ nặng đuôi.
3. Chỉ ra qua histogram, log-log survival plot, và QQ plot rằng hành vi đuôi rất khác nhau giữa các họ phân phối.
4. Định lượng khoảng cách khối lượng đuôi: sự kiện 4σ có xác suất 0.006% theo Gaussian nhưng phổ biến hơn nhiều theo mô hình đuôi nặng.

**Notebook tiếp theo:** `02_distribution_fitting.ipynb` — fit mô hình tham số vào các bộ dữ liệu này và so sánh theo AIC/BIC.